# Lesson 09 Lab — PTQ Calibration Data: Sampling and Coverage

**Puzzle:** Can a small calibration set represent the activation ranges that production traffic will exercise?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

A PTQ pipeline has a calibration distribution used to freeze quantization parameters and a disjoint evaluation distribution used to test the frozen result.

### Core mechanism

Max calibration protects observed extremes but can waste most codes; percentile or learned clipping trades a controlled tail for smaller steps. Either choice fails when the calibration set omits a deployment domain.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "09-ptq-calibration"
device = require_cuda()
torch.manual_seed(2026 + 9)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

More examples reduce sampling noise only when they add coverage. Long prompts, code, multilingual text, tool schemas, and rare outliers may need explicit strata rather than random repetition.

### What this code tests

The lab freezes scales from narrow, balanced, and outlier-aware sets and evaluates all three on one mixed held-out tensor.

**Experiment:** Calibrate INT8 activation scales on narrow, balanced, and outlier-aware synthetic datasets, then evaluate all scales on a mixed held-out distribution.

**Declared evidence label:** `numerical-model`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
def sample(n, mode):
    x = torch.randn(n, 512, device=device)
    if mode == "shifted": x = x * 2.5 + 1.5
    if mode == "rare": x[:, ::64] *= 10
    return x
eval_x = torch.cat([sample(1024,"base"), sample(512,"shifted"), sample(128,"rare")])
cal_sets = {"narrow": sample(1024,"base"), "balanced": torch.cat([sample(512,"base"),sample(512,"shifted")]),
            "outlier_aware": torch.cat([sample(448,"base"),sample(448,"shifted"),sample(128,"rare")])}
rows = {}
for name, cal in cal_sets.items():
    scale = cal.abs().max() / 127; q = torch.round(eval_x/scale).clamp(-128,127); dq=q*scale
    rows[name] = {"scale": round(scale.item(),8), "clipping_fraction": round((eval_x.abs()>127*scale).float().mean().item(),8),
                  "error": error_metrics(eval_x,dq)}
result=base_result(9,"numerical-model"); result.update({"evaluation_shape":list(eval_x.shape),"calibration_results":rows,
    "conclusion":"Held-out coverage, not calibration reconstruction, determined clipping and error."})


## 3. Inspect the evidence

Compare held-out clipping rate and error, not calibration-set reconstruction error.

### Acceptance and rollback gate

Publish sampling rules, lengths/domains, seed, statistic, sample count, and held-out clipping/error. Never tune the range on the same examples used for the final quality gate.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "calibration_results": {
    "balanced": {
      "clipping_fraction": 0.00025001,
      "error": {
        "cosine": 0.99894321,
        "mae": 0.0260779,
        "max_abs": 20.09401703,
        "rmse": 0.08504558
      },
      "scale": 0.10047337
    },
    "narrow": {
      "clipping_fraction": 0.02647752,
      "error": {
        "cosine": 0.98645717,
        "mae": 0.04251424,
        "max_abs": 27.9039135,
        "rmse": 0.31739485
      },
      "scale": 0.03945856
    },
    "outlier_aware": {
      "clipping_fraction": 0.0,
      "error": {
        "cosine": 0.99911618,
        "mae": 0.06723144,
        "max_abs": 0.13448334,
        "rmse": 0.07762861
      },
      "scale": 0.26896697
    }
  },
  "conclusion": "Held-out coverage, not calibration reconstruction, determined clipping and error.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
   

## 4. Explain the result

Choose calibration data by coverage of deployment modes, and keep it separate from the regression set.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).